# 🌱 智能温室监控系统 - 第一阶段：AI 模型训练

## 任务概述

在本阶段，你将使用 XEdu MMEdu 训练一个植物健康状态分类模型，能够识别植物的健康状况（健康、缺水、病害）。

### 学习目标
- ✅ 掌握图像分类的基本流程
- ✅ 学会使用 MMEdu 进行模型训练
- ✅ 理解数据增强和模型优化技术
- ✅ 能够评估和改进模型性能

### 积分奖励
- 完成基础训练：**+300 XP**
- 准确率 ≥ 90%：**+200 XP**
- 准确率 ≥ 95%：**+300 XP**
- 排行榜前 3 名：**+500 XP**

### 预计时长
4-6 小时

---

## 1️⃣ 准备工作

### 1.1 导入必要的库

In [ ]:
# 导入 XEdu MMEdu
from mmedu import ImageClassifier, Trainer
from mmedu.datasets import PlantHealthDataset
from mmedu.transforms import get_train_transforms, get_val_transforms

# 其他工具库
import os
import json
import matplotlib.pyplot as plt
%matplotlib inline

print("✅ 库导入成功！")

### 1.2 检查数据集

数据集结构:
```
data/plant_health/
├── train/
│   ├── healthy/      # 健康植物图片
│   ├── thirsty/      # 缺水植物图片
│   └── diseased/     # 病害植物图片
├── val/
│   ├── healthy/
│   ├── thirsty/
│   └── diseased/
└── test/
    ├── healthy/
    ├── thirsty/
    └── diseased/
```

In [ ]:
# 检查数据集
data_dir = 'data/plant_health'

if os.path.exists(data_dir):
    print(f"📁 数据集路径：{data_dir}")
    
    # 统计各类别数量
    categories = ['healthy', 'thirsty', 'diseased']
    category_names = {'healthy': '健康', 'thirsty': '缺水', 'diseased': '病害'}
    
    print("\n📊 数据集统计:")
    for split in ['train', 'val', 'test']:
        print(f"\n{split.upper()}集:")
        for cat in categories:
            cat_dir = os.path.join(data_dir, split, cat)
            if os.path.exists(cat_dir):
                count = len([f for f in os.listdir(cat_dir) if f.endswith(('.jpg', '.png'))])
                print(f"  - {category_names[cat]}: {count} 张")
else:
    print(f"❌ 数据集不存在：{data_dir}")
    print("💡 提示：请联系教师获取数据集或使用示例数据")

### 1.3 可视化样本图片

In [ ]:
# 显示每个类别的样本图片
def show_sample_images(data_dir, num_samples=3):
    """显示每个类别的样本图片"""
    categories = ['healthy', 'thirsty', 'diseased']
    category_names = {'healthy': '健康', 'thirsty': '缺水', 'diseased': '病害'}
    
    fig, axes = plt.subplots(3, num_samples, figsize=(15, 10))
    
    for i, cat in enumerate(categories):
        cat_dir = os.path.join(data_dir, 'train', cat)
        if not os.path.exists(cat_dir):
            continue
        
        images = [f for f in os.listdir(cat_dir) if f.endswith(('.jpg', '.png'))][:num_samples]
        
        for j, img_name in enumerate(images):
            img_path = os.path.join(cat_dir, img_name)
            img = plt.imread(img_path)
            
            if len(axes.shape) == 1:
                ax = axes[j]
            else:
                ax = axes[i, j]
            
            ax.imshow(img)
            ax.set_title(f'{category_names[cat]} - 样本{j+1}')
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# 显示样本
if os.path.exists(data_dir):
    show_sample_images(data_dir)

---

## 2️⃣ 构建和训练模型

### 2.1 配置数据增强

In [ ]:
# 定义数据增强策略
train_transforms = get_train_transforms(
    image_size=224,
    augmentations=[
        'random_horizontal_flip',  # 随机水平翻转
        'random_vertical_flip',    # 随机垂直翻转
        'random_rotation',         # 随机旋转 (-30°~30°)
        'random_crop',             # 随机裁剪
        'color_jitter',            # 颜色抖动
    ]
)

val_transforms = get_val_transforms(image_size=224)

print("✅ 数据增强配置完成")

### 2.2 加载数据集

In [ ]:
# 加载训练集和验证集
train_dataset = PlantHealthDataset(
    root=data_dir,
    split='train',
    transforms=train_transforms
)

val_dataset = PlantHealthDataset(
    root=data_dir,
    split='val',
    transforms=val_transforms
)

print(f"📊 训练集样本数：{len(train_dataset)}")
print(f"📊 验证集样本数：{len(val_dataset)}")

### 2.3 创建模型

In [ ]:
# 使用 ResNet-18 作为 backbone
model = ImageClassifier(
    backbone='resnet18',
    num_classes=3,  # 健康、缺水、病害
    pretrained=True  # 使用预训练权重
)

print("✅ 模型创建成功！")
print(f"📦 模型架构：ResNet-18")
print(f"🎯 输出类别数：3")

### 2.4 配置训练器

In [ ]:
# 配置训练参数
trainer = Trainer(
    model=model,
    epochs=20,              # 训练 20 轮
    batch_size=32,          # 批次大小
    learning_rate=0.001,    # 学习率
    criterion='cross_entropy',  # 损失函数
    optimizer='adam',       # 优化器
    metrics=['accuracy']    # 评估指标
)

print("✅ 训练器配置完成")
print(f"📈 训练轮数：20")
print(f"📦 批次大小：32")
print(f"💹 学习率：0.001")

### 2.5 开始训练

In [ ]:
# 开始训练
print("🚀 开始训练...")
print("="*60)

history = trainer.fit(
    train_dataset=train_dataset,
    val_dataset=val_dataset
)

print("="*60)
print("✅ 训练完成！")

### 2.6 可视化训练过程

In [ ]:
# 绘制准确率和损失曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 准确率曲线
ax1.plot(history['train_accuracy'], label='训练准确率', color='blue')
ax1.plot(history['val_accuracy'], label='验证准确率', color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('训练准确率变化')
ax1.legend()
ax1.grid(True)

# 损失曲线
ax2.plot(history['train_loss'], label='训练损失', color='blue')
ax2.plot(history['val_loss'], label='验证损失', color='red')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('损失函数变化')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# 打印最终指标
final_train_acc = history['train_accuracy'][-1]
final_val_acc = history['val_accuracy'][-1]

print(f"\n📊 最终训练结果:")
print(f"  - 训练准确率：{final_train_acc:.2%}")
print(f"  - 验证准确率：{final_val_acc:.2%}")
print(f"  - 训练轮数：{len(history['train_accuracy'])}")

---

## 3️⃣ 模型评估与保存

### 3.1 在测试集上评估

In [ ]:
# 加载测试集
test_dataset = PlantHealthDataset(
    root=data_dir,
    split='test',
    transforms=val_transforms
)

# 在测试集上评估
test_metrics = trainer.evaluate(test_dataset)

print(f"📊 测试集评估结果:")
print(f"  - 测试准确率：{test_metrics['accuracy']:.2%}")
print(f"  - 测试损失：{test_metrics['loss']:.4f}")

### 3.2 保存模型

In [ ]:
# 保存模型文件
import datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
model_filename = f'plant_health_classifier_{timestamp}.pth'

# 保存到 models 目录
os.makedirs('models', exist_ok=True)
model_path = os.path.join('models', model_filename)

model.save(model_path)
print(f"✅ 模型已保存：{model_path}")

# 同时保存为 ONNX 格式（可选，用于部署）
# model.export_onnx('models/plant_health_classifier.onnx')

### 3.3 生成训练报告

In [ ]:
# 生成训练报告
training_report = {
    'model_name': 'ResNet-18',
    'dataset': 'Plant Health Dataset',
    'epochs': 20,
    'batch_size': 32,
    'learning_rate': 0.001,
    'accuracy': float(final_val_acc),
    'test_accuracy': float(test_metrics['accuracy']),
    'loss': float(test_metrics['loss']),
    'training_time_minutes': len(history['train_accuracy']) * 1.5,  # 估算
    'model_path': model_path,
    'trained_at': datetime.datetime.now().isoformat()
}

# 保存报告
report_filename = f'training_report_{timestamp}.json'
with open(report_filename, 'w', encoding='utf-8') as f:
    json.dump(training_report, f, indent=2, ensure_ascii=False)

print(f"✅ 训练报告已保存：{report_filename}")
print("\n📄 报告内容:")
print(json.dumps(training_report, indent=2))

---

## 4️⃣ 提交作业

### 4.1 准备提交文件

In [ ]:
# 整理需要提交的文件
submission_files = [
    model_path,                    # 模型文件
    report_filename,               # 训练报告
    'notebooks/01_greenhouse_ai_training.ipynb'  # 本 Notebook
]

print("📦 请提交以下文件:")
for file in submission_files:
    print(f"  - {file}")

# 创建压缩包（可选）
import shutil
submission_dir = f'submission_{timestamp}'
os.makedirs(submission_dir, exist_ok=True)

# 复制文件到提交目录
for file in submission_files:
    if os.path.exists(file):
        shutil.copy(file, submission_dir)

# 压缩
shutil.make_archive(submission_dir, 'zip', submission_dir)
print(f"\n✅ 提交包已创建：{submission_dir}.zip")

### 4.2 上传到平台

🎉 **恭喜你完成第一阶段训练！**

现在你可以：

1. **通过 Web 界面提交**: 
   - 访问任务页面
   - 上传模型文件和训练报告
   - 系统会自动评测并发放积分

2. **通过 API 提交**:
```bash
curl -X POST \
  http://localhost:8000/api/v1/org/1/ai-edu/linked-tasks/greenhouse_001/stage/1/submit \
  -F "model_file=@models/plant_health_classifier.pth" \
  -F "training_report=@training_report.json"
```

### 积分计算
- 基础完成奖：**+300 XP**
- 准确率奖金（根据你的模型）:
  - ≥ 85%: +100 XP
  - ≥ 90%: +200 XP
  - ≥ 95%: +300 XP
- 排行榜前 3 名额外奖励：**+500 XP**

---

## 💡 进阶挑战（选做）

### 挑战 1: 提升模型准确率
尝试以下技术提升模型性能：
- 增加训练轮数（如 30 轮、50 轮）
- 调整学习率（学习率衰减）
- 使用更强的 backbone（如 ResNet-50、EfficientNet）
- 更多的数据增强
- 迁移学习微调

**奖励**: 准确率每提升 1%，获得 **+50 XP**

### 挑战 2: 模型轻量化
将模型压缩到 10MB 以内，同时保持准确率 > 85%

**奖励**: **+200 XP**

### 挑战 3: 实时推理优化
优化推理速度，达到每秒处理 10 张图片

**奖励**: **+150 XP**

---

## 📚 参考资料

- [XEdu MMEdu 官方文档](https://xedu.readthedocs.io/mmedu)
- [ResNet 论文](https://arxiv.org/abs/1512.03385)
- [图像分类实战教程](https://github.com/open-mmlab/mmclassification)
- [数据增强技巧](https://zhuanlan.zhihu.com/p/123456789)

---

**下一步**: 提交后进入第二阶段 - 硬件模拟集成 🚀